# Idea

The notebook analyses datasets for nulls, duplicates, and overlap.

Preprocessing steps:
- fix column naming schema (snake_case)
- fix column dtype
- remove rows with null
- remove duplicates
- remove overlap from normal dataset
- label data
- concatenate normal + fraud.

### Imports

In [18]:
from pathlib import Path

import pandas as pd
import numpy as np

### Paths

In [ ]:
PROJECT_ROOT = Path.cwd().parent.parent

DATA_PATH = PROJECT_ROOT / "data"

NORMAL_PATH = DATA_PATH / "raw_cdr" / "normal.parquet"
FRAUD_PATH = DATA_PATH / "raw_cdr" / "fraud.parquet"

CLEAN_CDR_PATH = DATA_PATH / "raw_cdr" / "clean_cdr.parquet"

FLAGGED_CELLS_PATH = DATA_PATH / "flagged_cells.npy"

---
### Load data

In [3]:
normal_df = pd.read_parquet(NORMAL_PATH)
fraud_df = pd.read_parquet(FRAUD_PATH)

print("Normal shape:", normal_df.shape)
print("Fraud shape :", fraud_df.shape)

Normal shape: (6849575, 8)
Fraud shape : (368754, 8)


---
### Data analysis

In [4]:
print("========== NORMAL ==========")
print("Shape:", normal_df.shape)
print("\nNulls:")
display(normal_df.isnull().sum())

print("\nDuplicates:")
print(normal_df.duplicated().sum())

print("\n========== FRAUD ==========")
print("Shape:", fraud_df.shape)
print("\nNulls:")
display(fraud_df.isnull().sum())

print("\nDuplicates:")
print(fraud_df.duplicated().sum())

common_rows = normal_df.merge(
    fraud_df,
    how="inner"
)

print("\n========== OVERLAP ==========")
print("Common rows:", len(common_rows))

========== NORMAL ==========
Shape: (6849575, 8)

Nulls:


MSISDN               0
AnswerTime           0
DurationInSeconds    0
LAC                  0
Cell                 0
CallingNumber        1
CalledNumber         0
RecordType           0
dtype: int64


Duplicates:
0

========== FRAUD ==========
Shape: (368754, 8)

Nulls:


MSISDN               0
AnswerTime           0
DurationInSeconds    0
LAC                  0
Cell                 0
CallingNumber        0
CalledNumber         0
RecordType           0
dtype: int64


Duplicates:
0


C:\Users\ali.farhat\AppData\Local\Temp\ipykernel_8972\789291603.py:17: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  common_rows = normal_df.merge(



========== OVERLAP ==========
Common rows: 5292


---
### Preprocessing function

In [5]:
def preprocess_datasets(normal_df, fraud_df):

    # ====================================
    # Copy
    # ====================================

    normal_df = normal_df.copy()
    fraud_df = fraud_df.copy()

    # ====================================
    # Column naming schema
    # ====================================

    rename_dict = {
        "MSISDN": "msisdn",
        "AnswerTime": "answer_time",
        "DurationInSeconds": "duration_in_seconds",
        "LAC": "lac",
        "Cell": "cell",
        "CallingNumber": "calling_number",
        "CalledNumber": "called_number",
        "RecordType": "record_type"
    }

    normal_df = normal_df.rename(columns=rename_dict)
    fraud_df = fraud_df.rename(columns=rename_dict)

    # ====================================
    # Datatypes
    # ====================================

    for df in [normal_df, fraud_df]:

        df["answer_time"] = pd.to_datetime(
            df["answer_time"],
            errors="coerce"
        )

        numeric_cols = [
            "msisdn",
            "duration_in_seconds",
            "lac",
            "cell",
            "calling_number",
            "called_number",
            "record_type"
        ]

        for col in numeric_cols:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    # ====================================
    # Remove null rows
    # ====================================

    normal_df = normal_df.dropna()
    fraud_df = fraud_df.dropna()

    # ====================================
    # Remove duplicates
    # ====================================

    normal_df = normal_df.drop_duplicates()
    fraud_df = fraud_df.drop_duplicates()

    # ====================================
    # Remove fraud rows from normal
    # ====================================

    normal_df = (
        normal_df
        .merge(
            fraud_df,
            how="left",
            indicator=True
        )
    )

    normal_df = (
        normal_df[
            normal_df["_merge"] == "left_only"
        ]
        .drop(columns="_merge")
    )

    # ====================================
    # Add labels
    # ====================================

    normal_df["label"] = 0
    fraud_df["label"] = 1

    # ====================================
    # Concatenate and shuffle
    # ====================================

    all_cdr_df = pd.concat(
        [normal_df, fraud_df],
        ignore_index=True
    )

    all_cdr_df = all_cdr_df.sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)

    return normal_df, fraud_df, all_cdr_df

### Apply preprocessing

In [6]:
normal_df, fraud_df, all_cdr_df = preprocess_datasets(
    normal_df,
    fraud_df
)

print("Normal shape:", normal_df.shape)
print("Fraud shape :", fraud_df.shape)
print("All CDR shape :", all_cdr_df.shape)

C:\Users\ali.farhat\AppData\Local\Temp\ipykernel_8972\1803691405.py:75: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  .merge(


Normal shape: (6844282, 9)
Fraud shape : (368754, 9)
All CDR shape : (7213036, 9)


### Verification

In [7]:
print("Normal NULLS:", normal_df.isnull().sum().sum())
print("Fraud NULLS :", fraud_df.isnull().sum().sum())

print("Normal duplicates:", normal_df.duplicated().sum())
print("Fraud duplicates :", fraud_df.duplicated().sum())

remaining_overlap = normal_df.merge(
    fraud_df,
    how="inner"
)

print("Remaining common rows:", len(remaining_overlap))

Normal NULLS: 0
Fraud NULLS : 0
Normal duplicates: 0
Fraud duplicates : 0


C:\Users\ali.farhat\AppData\Local\Temp\ipykernel_8972\4273659136.py:7: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  remaining_overlap = normal_df.merge(


Remaining common rows: 0


---
### Compute flagged cells

In [16]:
flagged_cells = (
    fraud_df["cell"]
    .drop_duplicates()
    .dropna()
    .astype("int64")
    .to_numpy()
)

print(f"Number of flagged cells: {flagged_cells.size}")

Number of flagged cells: 14404


---
### Save

In [19]:
np.save(
    FLAGGED_CELLS_PATH,
    flagged_cells
)

all_cdr_df.to_parquet(
    CLEAN_CDR_PATH,
    index=False
)

print("Dataset saved successfully.")

Dataset saved successfully.
